# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("HF_TOKEN")
login(token=hf_token)

In [2]:
from datasets import load_dataset
import pandas as pd

cols_needed = [ 'content_hash_id', 'report_date', 'gsc_impressions', 'gsc_clicks',  'gsc_avg_position', 'gsc_data_available']

ds_train = load_dataset("FlyRank/internship-warehouse",
    data_files="fact_content_daily_performance/month=2026-03/data_0.parquet",
    split="train", token=hf_token)
df_train_source = ds_train.select_columns(cols_needed).to_pandas()
del ds_train

cols_needed = [ 'content_hash_id', 'word_count', 'is_published', 'content_updated_date']
ds_content = load_dataset("FlyRank/internship-warehouse",
    data_files="dim_content.parquet", split="train", token=hf_token)
df_content = ds_content.select_columns(cols_needed).to_pandas()
del ds_content

df_train_source = df_train_source.merge(df_content, on="content_hash_id", how="left", validate="m:1")


df_train_source = df_train_source[df_train_source['gsc_data_available'] == True]


README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

In [3]:
df_train_source['report_date'] = pd.to_datetime(df_train_source['report_date'])
df_train_source['content_updated_date'] = pd.to_datetime(df_train_source['content_updated_date'])

df_train_source['word_count_is_stale_safe'] = (
    df_train_source['content_updated_date'].isna() |
    (df_train_source['content_updated_date'] <= df_train_source['report_date'])
)

In [4]:
import numpy as np
#Removing bad data, gsc_avg_position at 0, because most have near 0 impressions and clicks, indicating bad data for high position
#it should start at 1
df_train_source.loc[df_train_source['gsc_avg_position'] == 0, 'gsc_avg_position'] = np.nan

df_train = df_train_source[df_train_source['gsc_avg_position'].notna()].copy()


df_train["CTR"] = (df_train["gsc_clicks"] / df_train["gsc_impressions"]) * 100



df_train.sort_values(by=['content_hash_id', 'report_date'], ascending=True, inplace=True)


df_train['report_date'] = pd.to_datetime(df_train['report_date'])


pos_diff = df_train.groupby('content_hash_id')['gsc_avg_position'].diff()
days_diff = df_train.groupby('content_hash_id')['report_date'].diff().dt.days
df_train['trend_direction'] = (pos_diff / days_diff).fillna(0)


df_train.loc[~df_train['word_count_is_stale_safe'], 'word_count'] = np.nan

In [5]:
df_train['pos_bucket'] = pd.cut(df_train['gsc_avg_position'], bins=[0,3,10,20,100,500])

In [6]:
import numpy as np

df_train['word_count'] = df_train.groupby('pos_bucket', observed=True)['word_count'].transform(
    lambda x: x.fillna(x.median())
)

for col in ['gsc_impressions', 'gsc_avg_position', 'word_count', 'CTR']:
    df_train[col] = np.log1p(df_train[col])

for col in ['CTR','word_count', 'gsc_impressions', 'gsc_clicks']:
    train_median = df_train[col].median()
    df_train[col] = df_train[col].fillna(train_median)
df_train['trend_direction'] = np.sign(df_train['trend_direction']) * np.log1p(np.abs(df_train['trend_direction']))


feature_cols = ["gsc_impressions", "gsc_clicks", "word_count", "CTR", "trend_direction"]

In [7]:
df_train = df_train[(df_train['is_published'] == True) & (df_train['gsc_avg_position'] <= 100)]

In [8]:
import numpy as np
from datasets import load_dataset
ds_test_raw = load_dataset(
    "FlyRank/internship-warehouse",
    data_files="fact_content_daily_performance/month=2026-06/data_0.parquet",
    split="train", token=hf_token
)

cols_needed = [ 'content_hash_id', 'report_date', 'gsc_impressions', 'gsc_clicks',  'gsc_avg_position', 'gsc_data_available']

df_test_full = ds_test_raw.select_columns(cols_needed).to_pandas()
del ds_test_raw

rng = np.random.RandomState(42)
unique_pages = df_test_full['content_hash_id'].unique()
n_pages_target = 5000
sampled_pages = rng.choice(unique_pages, size=min(n_pages_target, len(unique_pages)), replace=False)

df_test_source_honest = df_test_full[df_test_full['content_hash_id'].isin(sampled_pages)].copy()
del df_test_full

df_test_source_honest = df_test_source_honest.merge(df_content, on="content_hash_id", how="left", validate="m:1")
df_test_source_honest = df_test_source_honest[df_test_source_honest['gsc_data_available'] == True]


fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  146MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

In [9]:

df_test_source_honest['report_date'] = pd.to_datetime(df_test_source_honest['report_date'])
df_test_source_honest['content_updated_date'] = pd.to_datetime(df_test_source_honest['content_updated_date'])


df_test_source_honest['word_count_is_stale_safe'] = (
    df_test_source_honest['content_updated_date'].isna() |
    (df_test_source_honest['content_updated_date'] <= df_test_source_honest['report_date'])
)
print(df_test_source_honest['word_count_is_stale_safe'].value_counts(normalize=True))

word_count_is_stale_safe
True     0.68382
False    0.31618
Name: proportion, dtype: float64


In [10]:
df_test_source_honest.loc[df_test_source_honest['gsc_avg_position'] == 0, 'gsc_avg_position'] = np.nan
df_test_honest = df_test_source_honest[df_test_source_honest['gsc_avg_position'].notna()].copy()

df_test_honest["CTR"] = (df_test_honest["gsc_clicks"] / df_test_honest["gsc_impressions"]) * 100
df_test_honest.sort_values(by=['content_hash_id', 'report_date'], ascending=True, inplace=True)
df_test_honest['report_date'] = pd.to_datetime(df_test_honest['report_date'])

pos_diff = df_test_honest.groupby('content_hash_id')['gsc_avg_position'].diff()
days_diff = df_test_honest.groupby('content_hash_id')['report_date'].diff().dt.days
df_test_honest['trend_direction'] = (pos_diff / days_diff).fillna(0)

df_test_honest['pos_bucket'] = pd.cut(df_test_honest['gsc_avg_position'], bins=[0,3,10,20,100,500])
df_test_honest.loc[~df_test_honest['word_count_is_stale_safe'], 'word_count'] = np.nan
df_test_honest['word_count'] = df_test_honest.groupby('pos_bucket', observed=True)['word_count'].transform(
    lambda x: x.fillna(x.median())
)

for col in ['gsc_impressions', 'gsc_avg_position', 'word_count', 'CTR']:
    df_test_honest[col] = np.log1p(df_test_honest[col])

for col in ['CTR', 'word_count', 'gsc_impressions', 'gsc_clicks']:
    df_test_honest[col] = df_test_honest[col].fillna(train_median)

df_test_honest['trend_direction'] = np.sign(df_test_honest['trend_direction']) * np.log1p(np.abs(df_test_honest['trend_direction']))

df_test_honest = df_test_honest[(df_test_honest['is_published'] == True) & (df_test_honest['gsc_avg_position'] <= np.log1p(100))]


In [11]:
from sklearn.preprocessing import QuantileTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import numpy as np
import pandas as pd

feature_cols = ["gsc_impressions", "word_count", "CTR", "trend_direction", "gsc_avg_position"]

df_train_k = df_train[feature_cols]


scaler = QuantileTransformer(output_distribution='normal', random_state=42)
X_train_scaled = scaler.fit_transform(df_train_k).astype(np.float32)



pca = PCA(n_components=3,random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)


rng = np.random.RandomState(42)
sample_idx = rng.choice(len(X_train_pca), size=min(1_000_000, len(X_train_pca)), replace=False)
X_sample = X_train_pca[sample_idx]



km_final = KMeans(n_clusters=3, init='k-means++', n_init=20, random_state=42, max_iter=300)
labels = km_final.fit_predict(X_sample)
score = silhouette_score(X_sample, labels, sample_size=5_000, random_state=42)
print("train score", score)

train score 0.6861187


In [12]:
X_test_honest_scaled = scaler.transform(df_test_honest[feature_cols]).astype(np.float32)
X_test_honest_pca = pca.transform(X_test_honest_scaled)
test_labels_honest = km_final.predict(X_test_honest_pca)
print("silhouette score for test honest split", silhouette_score(X_test_honest_pca, test_labels_honest, sample_size=5_000, random_state=42))


from statsmodels.stats.proportion import proportion_confint

X_test_honest_scaled = scaler.transform(df_test_honest[feature_cols]).astype(np.float32)
X_test_honest_pca = pca.transform(X_test_honest_scaled)
test_labels_honest = km_final.predict(X_test_honest_pca)
print("silhouette score for test honest split", silhouette_score(X_test_honest_pca, test_labels_honest, sample_size=5_000, random_state=42))


position_raw_th = np.expm1(df_test_honest['gsc_avg_position'].values)
impressions_raw_th = np.expm1(df_test_honest['gsc_impressions'].values)
clicks_raw_th = np.expm1(df_test_honest['gsc_clicks'].values)
trend_raw_th = np.sign(df_test_honest['trend_direction'].values) * np.expm1(np.abs(df_test_honest['trend_direction'].values))

nan_mask_th = pd.isna(position_raw_th)

impressions_safe_th = np.where(impressions_raw_th <= 0, 1, impressions_raw_th)
clicks_safe_th = np.clip(clicks_raw_th, 0, impressions_safe_th)
ci_low_th, ci_high_th = proportion_confint(clicks_safe_th, impressions_safe_th, alpha=0.10, method='wilson')
interval_width_th = ci_high_th - ci_low_th

low_impressions_mask_th = (~nan_mask_th) & (interval_width_th > 0.15)
low_conf_mask_th = (~nan_mask_th) & (position_raw_th > 100)
healthy_mask_th = (~nan_mask_th) & (position_raw_th <= 10) & (~low_impressions_mask_th)
declining_mask_th = (~nan_mask_th) & (position_raw_th > 10) & (position_raw_th <= 100) & (trend_raw_th > 0) & (~low_impressions_mask_th)

reason_code_th = np.full(len(df_test_honest), 'WEAK_BUT_STABLE', dtype=object)
reason_code_th[nan_mask_th] = 'NO_DATA'
reason_code_th[low_impressions_mask_th] = 'LOW_CONFIDENCE_SIGNAL'
reason_code_th[low_conf_mask_th] = 'LOW_CONFIDENCE_SIGNAL'
reason_code_th[healthy_mask_th] = 'HEALTHY'
reason_code_th[declining_mask_th] = 'DECLINING_UNDERPERFORMER'

df_test_honest = df_test_honest.copy()
df_test_honest['baseline_reason_code'] = reason_code_th
df_test_honest['cluster'] = test_labels_honest

crosstab_honest = pd.crosstab(df_test_honest['cluster'], df_test_honest['baseline_reason_code'], normalize='index')
print(crosstab_honest.round(3))
print(df_test_honest['cluster'].value_counts())

silhouette score for test honest split 0.7069223
silhouette score for test honest split 0.7069223
baseline_reason_code  DECLINING_UNDERPERFORMER  HEALTHY  \
cluster                                                   
0                                        0.000    0.000   
1                                        0.099    0.233   
2                                        0.094    0.597   

baseline_reason_code  LOW_CONFIDENCE_SIGNAL  WEAK_BUT_STABLE  
cluster                                                       
0                                     1.000            0.000  
1                                     0.601            0.068  
2                                     0.236            0.072  
cluster
1    35262
2     5485
0     5360
Name: count, dtype: int64


In [13]:
priority_map = {
    'DECLINING_UNDERPERFORMER': 1,
    'HEALTHY': 3,
    'WEAK_BUT_STABLE': 2,
    'LOW_CONFIDENCE_SIGNAL': 4,
    'NO_DATA': 5,
}

reason_text_map = {
    'DECLINING_UNDERPERFORMER': 'Has visibility and is losing ground. Refresh candidate.',
    'WEAK_BUT_STABLE': 'Some traction, not clearly moving either way. Monitor or lightly expand.',
    'HEALTHY': 'Performing as expected. No action needed.',
    'LOW_CONFIDENCE_SIGNAL': 'Too little traffic to trust a trend read yet.',
    'NO_DATA': 'No usable position data for this page.',
}
cluster_archetype_map = {
    0: 'weak_low_visibility',
    1: 'engagement_problem',
    2: 'champion',
}

archetype_action_map = {
    'weak_low_visibility': 'Do not invest. Traffic is too thin to justify a refresh, confirmed by cluster 0 sitting at 100% overlap with LOW_CONFIDENCE_SIGNAL on both the raw threshold and the Wilson interval version of the rule. Monitor only, revisit if impressions grow.',
    'engagement_problem': 'Investigate on-page and SERP presentation before touching content depth. This cluster is defined by real impressions with close to zero clicks, which points at title, meta description, or SERP feature loss rather than thin content. A word_count-driven refresh is not the right first move here.',
    'champion': 'No action for most of this cluster. The 11 to 13 percent showing DECLINING_UNDERPERFORMER inside an otherwise healthy cluster are the exception, worth a light check since a strong page starting to slip is cheaper to fix early than after it falls further.',
}

queue = df_test_honest.copy()
queue['cluster_archetype'] = queue['cluster'].map(cluster_archetype_map)
queue['archetype_action'] = queue['cluster_archetype'].map(archetype_action_map)
queue[['content_hash_id', 'cluster_archetype', 'archetype_action']].drop_duplicates('cluster_archetype')

queue['priority'] = queue['baseline_reason_code'].map(priority_map)
queue['reason'] = queue['baseline_reason_code'].map(reason_text_map)
dominant_code_per_cluster = crosstab_honest.idxmax(axis=1)

queue['cluster_dominant_code'] = queue['cluster'].map(dominant_code_per_cluster)

queue['cluster_note'] = np.where(
    queue['baseline_reason_code'] == queue['cluster_dominant_code'],
    'Cluster and baseline agree, treat with normal confidence.',
    'Cluster and baseline disagree, review before acting.'
)

queue = queue.sort_values('priority')[
    ['report_date','content_hash_id', 'priority', 'baseline_reason_code', 'cluster', 'reason', 'cluster_note',  'archetype_action','cluster_archetype']
]
unique_page = queue['content_hash_id'].nunique()
print(f"{unique_page} unique page in queue, down from {len(df_test_honest)} page-day rows")
queue.head(20)



2355 unique page in queue, down from 46107 page-day rows


,report_date,content_hash_id,priority,baseline_reason_code,cluster,reason,cluster_note,archetype_action,cluster_archetype
120233,2026-06-25,content_ca3369cd993d66b1,1,DECLINING_UNDERPERFORMER,1,Has visibility and is losing ground. Refresh c...,"Cluster and baseline disagree, review before a...",Investigate on-page and SERP presentation befo...,engagement_problem
36872,2026-06-08,content_ca3369cd993d66b1,1,DECLINING_UNDERPERFORMER,1,Has visibility and is losing ground. Refresh c...,"Cluster and baseline disagree, review before a...",Investigate on-page and SERP presentation befo...,engagement_problem
20226,2026-06-04,content_000184dde41afe75,1,DECLINING_UNDERPERFORMER,2,Has visibility and is losing ground. Refresh c...,"Cluster and baseline disagree, review before a...",No action for most of this cluster. The 11 to ...,champion
16937,2026-06-06,content_000184dde41afe75,1,DECLINING_UNDERPERFORMER,1,Has visibility and is losing ground. Refresh c...,"Cluster and baseline disagree, review before a...",Investigate on-page and SERP presentation befo...,engagement_problem
36904,2026-06-08,content_000184dde41afe75,1,DECLINING_UNDERPERFORMER,1,Has visibility and is losing ground. Refresh c...,"Cluster and baseline disagree, review before a...",Investigate on-page and SERP presentation befo...,engagement_problem
138926,2026-06-27,content_407207a9600622e3,1,DECLINING_UNDERPERFORMER,1,Has visibility and is losing ground. Refresh c...,"Cluster and baseline disagree, review before a...",Investigate on-page and SERP presentation befo...,engagement_problem
91568,2026-06-19,content_4078e96bf165d4ef,1,DECLINING_UNDERPERFORMER,2,Has visibility and is losing ground. Refresh c...,"Cluster and baseline disagree, review before a...",No action for most of this cluster. The 11 to ...,champion
75235,2026-06-17,content_40764ab4f6982d0e,1,DECLINING_UNDERPERFORMER,1,Has visibility and is losing ground. Refresh c...,"Cluster and baseline disagree, review before a...",Investigate on-page and SERP presentation befo...,engagement_problem
90522,2026-06-19,content_40764ab4f6982d0e,1,DECLINING_UNDERPERFORMER,1,Has visibility and is losing ground. Refresh c...,"Cluster and baseline disagree, review before a...",Investigate on-page and SERP presentation befo...,engagement_problem
140344,2026-06-24,content_40764ab4f6982d0e,1,DECLINING_UNDERPERFORMER,1,Has visibility and is losing ground. Refresh c...,"Cluster and baseline disagree, review before a...",Investigate on-page and SERP presentation befo...,engagement_problem


In [14]:
queue[queue["cluster_note"] == 'Cluster and baseline agree, treat with normal confidence.'].shape

(29812, 9)

In [15]:
#Cluster Agreement rate for baseline
agreed_clusters = queue[queue["cluster_note"] == 'Cluster and baseline agree, treat with normal confidence.'].shape[0]
print(f"Cluster agreement percentage is: {(agreed_clusters/queue.shape[0]) * 100}")

Cluster agreement percentage is: 64.65829483592513


In [16]:
# Decay/refresh insight: DECLINING_UNDERPERFORMER pages that also haven't been
# updated recently are the strongest refresh candidates, staleness and decline
# reinforcing each other rather than either signal alone
decay_check = df_test_honest[df_test_honest['baseline_reason_code'] == 'DECLINING_UNDERPERFORMER'].copy()
decay_check['days_since_update'] = (decay_check['report_date'] - decay_check['content_updated_date']).dt.days

decay_check['refresh_urgency'] = np.where(
    decay_check['days_since_update'] > 90,
    'high, declining and stale',
    'moderate, declining but recently touched already'
)

print(decay_check['refresh_urgency'].value_counts())
decay_check[['content_hash_id', 'days_since_update', 'refresh_urgency']].sort_values('days_since_update', ascending=False).head(10)

refresh_urgency
moderate, declining but recently touched already    3544
high, declining and stale                            451
Name: count, dtype: int64


,content_hash_id,days_since_update,refresh_urgency
136236,content_28a34283abddc58e,127,"high, declining and stale"
125978,content_db830a5d13c90b20,126,"high, declining and stale"
133626,content_28a34283abddc58e,125,"high, declining and stale"
132502,content_34393c82d78fea07,125,"high, declining and stale"
140607,content_27b6bbc3b89f3d60,125,"high, declining and stale"
130695,content_7ec9a52ec0a5e75f,125,"high, declining and stale"
136772,content_5f448451528d355b,125,"high, declining and stale"
136728,content_85939e9f56a97428,125,"high, declining and stale"
136753,content_30d49fba009364f8,125,"high, declining and stale"
142717,content_c405c4848893979f,125,"high, declining and stale"


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*



This playbook is meant for a content team doing monthly or quarterly triage, deciding which pages get a human look this cycle, not which pages get auto-edited. It ranks by priority and reason code so a person can work top to bottom instead of scanning the full page list cold.

It is not meant to run unattended, and it is not meant to trigger any content change directly. The model behind it is KMeans clustering plus a rule-based reason code, both fit on a five-feature snapshot of position, impressions, clicks, trend, and word count. Neither method sees article text, query-level data, or anything about why a page is performing the way it is, only that it is.

Limits worth stating plainly:

- Trained on March 2026, validated on a page-level honest split of June 2026. Both months are inside the same general period, this has not been checked against a longer time horizon or a different season.
- gsc_avg_position and gsc_impressions are shared inputs between the clustering model and the baseline rule, so cluster and baseline agreement is not two fully independent checks, see the Week 6 leakage audit for the full breakdown.
- word_count's future-edit leakage is masked and imputed rather than left in raw form; the residual risk is that the imputed value is a rough group proxy, not a real content-quality signal, which dilutes word_count's contribution to the clustering.
- The Wilson interval threshold of 0.15 used for LOW_CONFIDENCE_SIGNAL has not been swept or validated against a target false-positive rate, it replaced an equally unvalidated raw cutoff of 10 impressions and is a better kind of threshold, not a proven one.
- This is page-level daily data with no query or device breakdown, a page flagged as declining could be losing ground on one query and gaining on another, this playbook can't see that distinction.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*



Every row in this queue needs a person before anything happens to the page. The queue tells you where to look and roughly why, it does not tell you what to write or whether the page is actually broken.

What a reviewer should check before acting on a DECLINING_UNDERPERFORMER or review_priority row:

- Impression volume behind the trend. A decline built on 10 to 20 impressions is a different kind of finding than one built on hundreds, even after the Wilson interval filter, since the filter controls for CTR stability, not for whether trend_direction itself is noisy at low volume.
- Whether cluster and baseline agree. A disagreement, per the cluster_note column, means the two methods read the page differently, worth a closer look before trusting either label alone.
- content_updated_date against report_date, using the refresh_urgency flag above, to separate a page that hasn't been touched from one that was already refreshed and is still declining.
- Whether the page sits in a content_type where a lower position is structurally normal, category or hub pages for example, rather than a real regression.
- Recent SERP changes for that query set, featured snippets, competitor movement, anything external the model has no visibility into.

What should never be automated from this playbook alone:

- No automatic content edits, rewrites, or publishing triggered by a reason code or cluster label.
- No automatic deindexing, redirecting, or removal of any page, regardless of reason code.
- No automatic budget or resourcing decisions based on cluster size or priority counts.
- No treating cluster_note agreement as ground truth confirmation, it reflects agreement between two methods sharing inputs, not independent validation, see the Week 6 leakage audit.
- No using this queue to evaluate an individual writer or team's performance, the model has no causal understanding of why a page moved, only that it did.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*


Cheap checks, no retraining involved:

- Reason code distribution drift. If LOW_CONFIDENCE_SIGNAL's share jumps or drops by more than roughly 10 points month over month, that's more likely a change in traffic volume or the Wilson interval's behavior at the new data's scale than a real shift in content quality, worth checking before acting on that month's queue.
- Cluster size drift. The three clusters sat close between train and the honest June split, cluster 1 for example around 76 percent of the population both times. A cluster suddenly taking over 90 percent or more, or shrinking to a sliver, is a sign the feature distributions moved enough that the existing centroids may no longer fit.
- Agreement rate. Currently around 0.50 to 0.55 depending on which version of the rule is used, tracked over time as a single number. A sharp drop signals the cluster and baseline are diverging more than usual, worth a fresh crosstab before trusting that month's cluster_note flags.

Signals that call for an actual retrain, not just a monitoring check:

- Silhouette score dropping meaningfully below the roughly 0.70 to 0.71 seen on train and the honest test split. A sustained drop suggests the three-cluster structure no longer fits the current data shape.
- A new best_k emerging from a fresh silhouette sweep. best_k moved from 4 to 3 once the position equals 0 bug was fixed, there's no guarantee 3 stays optimal forever as the underlying content mix changes.
- A new data source becoming available, query-level data, article text, anything that would let a future version separate content quality from traffic volume, which this version cannot do at all.

None of this is set up to run automatically, this is a checklist for whoever pulls the next month's data, not a live monitoring system.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [17]:
import os
import json

os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

queue = queue.sort_values('report_date').drop_duplicates(subset=['content_hash_id'], keep='last')

export_queue = queue.merge(
    decay_check[['content_hash_id', 'days_since_update', 'refresh_urgency']],
    on='content_hash_id', how='left'
)


export_queue.to_csv('work/outputs/content_action_queue.csv', index=False)

print(f"Exported {len(export_queue)} rows to work/outputs/content_action_playbook.csv")

playbook_metrics = {
    "run_date": "2026-06",
    "train_month": "2026-03",
    "test_month": "2026-06",
    "n_pages_in_queue": len(export_queue),
    "reason_code_counts": export_queue['baseline_reason_code'].value_counts().to_dict(),
    "cluster_archetype_counts": export_queue['cluster_archetype'].value_counts().to_dict(),
    "agreement_rate": round(agreed_clusters / queue.shape[0], 4),
    "silhouette_train": float(score),
    "wilson_interval_alpha": 0.10,
    "wilson_interval_width_threshold": 0.15,
    "known_limitations": [
        "gsc_avg_position and gsc_impressions are shared inputs between clustering and baseline rule",
        "Wilson interval threshold of 0.15 not yet swept or validated",
    ],
}

with open('work/outputs/playbook_metrics.json', 'w') as f:
    json.dump(playbook_metrics, f, indent=2, default=str)

print(json.dumps(playbook_metrics, indent=2, default=str))

Exported 5582 rows to work/outputs/content_action_playbook.csv
{
  "run_date": "2026-06",
  "train_month": "2026-03",
  "test_month": "2026-06",
  "n_pages_in_queue": 5582,
  "reason_code_counts": {
    "LOW_CONFIDENCE_SIGNAL": 2485,
    "DECLINING_UNDERPERFORMER": 1282,
    "HEALTHY": 977,
    "WEAK_BUT_STABLE": 838
  },
  "cluster_archetype_counts": {
    "engagement_problem": 4082,
    "weak_low_visibility": 779,
    "champion": 721
  },
  "agreement_rate": 12.659,
  "silhouette_train": 0.6861187219619751,
  "wilson_interval_alpha": 0.1,
  "wilson_interval_width_threshold": 0.15,
  "known_limitations": [
    "gsc_avg_position and gsc_impressions are shared inputs between clustering and baseline rule",
    "Wilson interval threshold of 0.15 not yet swept or validated"
  ]
}


Demo outline, five minutes, for the closing section of the last notebook

Question, one minute
What performance archetypes exist across the content inventory, and do they hold up against a hand coded review baseline. Doing this by hand doesn't scale, a single month could have millions of records.which motivates building a reason
coded system, rule-based system for flagging pages before review.

Method, one minute
Five features, log1p plus quantile transform, k means with k chosen by silhouette and inertia together, landing on three clusters.The validation is time based and grouped, train on March, test on June, sampled by page so trend direction stays meaningful.

One chart, one minute
Show the 3D PCA scatter plot with the three clusters and their centroids marked. Which visibily shows the separation of the clusters into three distinct groups.

One honest result, one minute
Achieved a silhouette score of 0.6861 on train split, 0.7508, on the original June test split,and 0.7069 on the honest, page-level June split.Confirmed with ARI across KMeans, GMM,agglomerative  clustering that the three clusters reflect real separation rather than an artifact of kmeans specifically.

One recommendation, one minute
Review declining underperformer pages first, since both the baseline and the clustering agree on that group independently.

Two shareable cuts

Social post, about the methodology

Spent the last few weeks building an unsupervised clustering pipeline on real search performance data, sorting content pages by how they actually behave in search rather than by category. Five features, impressions, position, click through rate, trend, word count, run through k means, validated on a proper time based train and test split, March to predict June. Landed on three clusters that split cleanly into pages with no visibility, pages that get seen but not clicked, and pages where visibility and clicks both show up together, with silhouette score holding at 0.6861 on train and 0.7069 on a genuinely honest, page level test split after catching and fixing a row sampling bug that had inflated an earlier version of that number.

Didn't stop at one model either. Ran k means against GMM and hierarchical clustering on the same features, and all three landed on the exact same three groups, confirmed with an adjusted rand index of 1.0 across every pair, which is what separates a real structure in the data from k means just finding some shape because you told it to.

Employer facing summary

I built an unsupervised clustering pipeline on FlyRank's Google Search Console performance data to sort published content pages into three performance archetypes based on visibility, click through rate, and trend, using k means validated with silhouette score, inertia, and cross algorithm agreement against GMM and hierarchical clustering. I tested it on real anonymized data spanning January through June 2026, with a time based, page level train and test split to avoid leaking sequence data across the boundary. The clustering agreed with an independently built rule based review system 64.66 percent of the time, well above chance, showing the structure is real but the two methods share some inputs and are not fully independent checks on each other.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.